In [1]:
# Test Neo4j connection
from neo4j import GraphDatabase

uri = "bolt://localhost:7687"
user = "neo4j"
pwd = "your_local_password"

try:
    driver = GraphDatabase.driver(uri, auth=(user, pwd))
    with driver.session() as session:
        result = session.run("RETURN 'Hello, Neo4j!' AS message")
        record = result.single()
        print(record["message"])
    driver.close()
    print("Connection successful!")
except Exception as e:
    print(f"Connection failed: {e}")

Connection failed: {neo4j_code: Neo.ClientError.Security.Unauthorized} {message: The client is unauthorized due to authentication failure.} {gql_status: 42NFF} {gql_status_description: error: syntax error or access rule violation - permission/access denied. Access denied, see the security logs for details.}


In [2]:
# Query the knowledge graph
from neo4j import GraphDatabase
import os
import dotenv

dotenv.load_dotenv()

uri = "bolt://localhost:7687"
user = "neo4j"
pwd = os.getenv("password")

driver = GraphDatabase.driver(uri, auth=(user, pwd))

with driver.session() as session:
    # Count nodes
    result = session.run("MATCH (n) RETURN labels(n) as labels, count(*) as count")
    print("Node counts:")
    for record in result:
        print(f"  {record['labels']}: {record['count']}")
    
    # Check for descriptions
    result = session.run("MATCH (a:ArtPiece) WHERE a.description IS NOT NULL RETURN a.title as title, a.description as description LIMIT 5")
    print("\nArtworks with descriptions:")
    for record in result:
        print(f"  {record['title']}: {record['description'][:100]}...")

driver.close()

Node counts:
  ['Palau']: 3
  ['Sala']: 6
  ['ArtPiece']: 72
  ['Artist']: 23
  ['Theme']: 38

Artworks with descriptions:
  Sant Jeroni penitent: (Entre les taules de Sant Jeroni i els Ressuscitats) Aquestes dues pintures sobre taula ens ajuden a...
  Davallament: Moltes de les obres d’art que avui veiem dins d’un museu han perdut el context i la funció originals...
  Mare de Déu de la Llet: Moltes de les obres d’art que avui veiem dins d’un museu han perdut el context i la funció originals...
  Flagel·lació de Crist: Dins del conjunt destaca una escultura en fusta policromada. La imatge de la Mare de Déu de Trapani ...


In [6]:
# Add room descriptions to the knowledge graph
from docx import Document
from neo4j import GraphDatabase

def read_docx(file_path):
    doc = Document(file_path)
    text = ""
    for para in doc.paragraphs:
        if para.text.strip():
            text += para.text + "\n"
    return text

def add_room_descriptions(uri, user, pwd):
    room_texts = read_docx("../raw_data/Textos_de_sala_MUSEU_DEL_RENAIXEMENT.docx")

    driver = GraphDatabase.driver(uri, auth=(user, pwd))

    # Parse room texts - they seem to be labeled as "Text X – PB Y" or similar
    sections = room_texts.split("Text ")
    for section in sections[1:]:  # Skip the header
        lines = section.split('\n')
        if lines:
            header = lines[0].strip()
            content = '\n'.join(lines[1:]).strip()
            
            # Try to extract room info, e.g., "1 – PB 0" or "2 – PB 0"
            if ' – ' in header:
                text_num, location = header.split(' – ', 1)
                if 'PB' in location or 'P' in location:
                    # Map PB 0 to palau 1, sala 0 or something
                    # Assuming PB means Planta Baixa (Ground Floor), perhaps palau 1
                    if 'PB 0' in location:
                        palau_id = '1'
                        sala_id = '0'
                    elif 'PB 1' in location:
                        palau_id = '1'
                        sala_id = '1'
                    else:
                        continue
                    
                    with driver.session() as session:
                        session.run("""
                        MATCH (s:Sala {id: $sala, palau: $palau})
                        SET s.description = $description
                        """, sala=sala_id, palau=palau_id, description=content)
                    print(f"Added description to Palau {palau_id}, Sala {sala_id}")

    driver.close()
    print("Room descriptions added!")

# Run the function
add_room_descriptions("bolt://localhost:7687", "neo4j", pwd)

Added description to Palau 1, Sala 0
Added description to Palau 1, Sala 0
Added description to Palau 1, Sala 1
Room descriptions added!


In [7]:
# Improved loading with Artist nodes
import pandas as pd
from neo4j import GraphDatabase

def parse_ubic(ubic_str):
    if not isinstance(ubic_str, str) or '-' not in ubic_str:
        return "Unknown", "Unknown"
    parts = ubic_str.split('-')
    palau = parts[0].replace('P', '')
    sala = parts[1].replace('S', '')
    return palau, sala

def load_museum_excel_improved(file_path, uri, user, pwd):
    df = pd.read_excel(file_path, header=None)
    df = df[1:]  # skip title row
    df.columns = df.iloc[0]  # set headers
    df = df[1:]  # skip header row
    
    driver = GraphDatabase.driver(uri, auth=(user, pwd))
    
    with driver.session() as session:
        for _, row in df.iterrows():
            palau_id, sala_id = parse_ubic(row['UBIC.'])
            artist_name = row['AUTORIA']
            
            # Create ArtPiece
            session.run("""
            MERGE (p:Palau {id: $palau})
            MERGE (s:Sala {id: $sala, palau: $palau})
            MERGE (p)-[:HAS_SALA]->(s)
            
            MERGE (a:ArtPiece {title: $title})
            SET a.artist = $artist, 
                a.dating = $dating,
                a.technique = $technique
            MERGE (a)-[:LOCATED_IN]->(s)
            """, 
            palau=palau_id, sala=sala_id, 
            title=row['TÍTOL / DESCRIPCIÓ'], 
            artist=artist_name,
            dating=row['DATACIÓ'],
            technique=row['TÈCNICA / TIPOLOGIA'])
            
            # Link to artist if not anonymous
            if pd.notna(artist_name) and 'Anònim' not in artist_name:
                session.run("""
                MERGE (art:Artist {name: $name})
                MERGE (a:ArtPiece {title: $title})
                MERGE (a)-[:CREATED_BY]->(art)
                """, name=artist_name, title=row['TÍTOL / DESCRIPCIÓ'])
    
    # Create relationships between artists
    with driver.session() as session:
        session.run("""
        MATCH (a1:Artist)-[:CREATED_BY]-(art1:ArtPiece), (a2:Artist)-[:CREATED_BY]-(art2:ArtPiece)
        WHERE a1 <> a2 AND art1.dating = art2.dating AND art1.dating IS NOT NULL
        MERGE (a1)-[:CONTEMPORARY_OF {period: art1.dating}]->(a2)
        """)
    
    driver.close()
    print("Improved data loaded with Artist nodes and relationships!")

# Run the improved loading
load_museum_excel_improved("/Users/neromelt/UAB/Synthesys II/GuIA/raw_data/2026_obres_Museu_del_Renaixement.xlsx", 
                           "bolt://localhost:7687", "neo4j", pwd)

Improved data loaded with Artist nodes and relationships!


In [8]:
# Create Theme nodes from audio guide
from docx import Document
from neo4j import GraphDatabase

def read_docx(file_path):
    doc = Document(file_path)
    text = ""
    for para in doc.paragraphs:
        if para.text.strip():
            text += para.text + "\n"
    return text

def create_themes_from_audio_guide(uri, user, pwd):
    audio_text = read_docx("../raw_data/Guió_Audioguia_Museu_Renaixement.docx")
    
    driver = GraphDatabase.driver(uri, auth=(user, pwd))
    
    # Extract sections as themes
    lines = audio_text.split('\n')
    current_theme = None
    theme_content = ""
    
    for line in lines:
        line = line.strip()
        if line and not line.startswith(' ') and len(line) < 100:  # Likely a section title
            if current_theme:
                # Save previous theme
                with driver.session() as session:
                    session.run("""
                    MERGE (t:Theme {name: $name})
                    SET t.description = $description
                    """, name=current_theme, description=theme_content.strip())
            
            current_theme = line
            theme_content = ""
        else:
            theme_content += line + " "
    
    # Save last theme
    if current_theme:
        with driver.session() as session:
            session.run("""
            MERGE (t:Theme {name: $name})
            SET t.description = $description
            """, name=current_theme, description=theme_content.strip())
    
    # Link themes to rooms based on content (simple approach)
    with driver.session() as session:
        # For example, link "Primera planta" themes to palau 1
        session.run("""
        MATCH (t:Theme) WHERE t.name CONTAINS 'Primera'
        MATCH (p:Palau {id: '1'})
        MERGE (p)-[:FEATURES_THEME]->(t)
        """)
        
        session.run("""
        MATCH (t:Theme) WHERE t.name CONTAINS 'Planta baixa'
        MATCH (p:Palau {id: '1'})
        MERGE (p)-[:FEATURES_THEME]->(t)
        """)
    
    driver.close()
    print("Themes created from audio guide!")

# Run theme creation
create_themes_from_audio_guide("bolt://localhost:7687", "neo4j", pwd)

Themes created from audio guide!


In [9]:
# PART 1: Access via NEO4J BROWSER (Visual Interface)
# ================================================
# 1. Open: http://localhost:7474
# 2. Login with:
#    Username: neo4j
#    Password: (your local password from .env)
# 3. Copy-paste any of the Cypher queries below directly into the browser

cypher_examples = {
    "View all nodes": """
    MATCH (n) RETURN n LIMIT 50
    """,
    
    "All artworks": """
    MATCH (a:ArtPiece) RETURN a.title, a.artist, a.dating, a.technique
    """,
    
    "Artworks by location": """
    MATCH (a:ArtPiece)-[:LOCATED_IN]->(s:Sala)-[:HAS_ROOM*0..]->(p:Palau)
    RETURN p.id as palau, s.id as sala, a.title, a.artist
    """,
    
    "Artists and their works": """
    MATCH (artist:Artist)-[:CREATED_BY]-(artwork:ArtPiece)
    RETURN artist.name, COUNT(artwork) as num_works
    ORDER BY num_works DESC
    """,
    
    "Contemporary artists": """
    MATCH (a1:Artist)-[r:CONTEMPORARY_OF]->(a2:Artist)
    RETURN a1.name, a2.name, r.period
    """,
    
    "Rooms with descriptions": """
    MATCH (s:Sala) WHERE s.description IS NOT NULL
    RETURN s.palau, s.id, s.description
    """,
    
    "Themes in each palace": """
    MATCH (p:Palau)-[:FEATURES_THEME]->(t:Theme)
    RETURN p.id as palau, t.name as theme
    """
}

print("📊 NEO4J BROWSER - Example Cypher Queries")
print("=" * 60)
print("\nOpen http://localhost:7474 and try these queries:\n")
for title, query in cypher_examples.items():
    print(f"• {title}:")
    print(f"  {query.strip()}\n")


📊 NEO4J BROWSER - Example Cypher Queries

Open http://localhost:7474 and try these queries:

• View all nodes:
  MATCH (n) RETURN n LIMIT 50

• All artworks:
  MATCH (a:ArtPiece) RETURN a.title, a.artist, a.dating, a.technique

• Artworks by location:
  MATCH (a:ArtPiece)-[:LOCATED_IN]->(s:Sala)-[:HAS_ROOM*0..]->(p:Palau)
    RETURN p.id as palau, s.id as sala, a.title, a.artist

• Artists and their works:
  MATCH (artist:Artist)-[:CREATED_BY]-(artwork:ArtPiece)
    RETURN artist.name, COUNT(artwork) as num_works
    ORDER BY num_works DESC

• Contemporary artists:
  MATCH (a1:Artist)-[r:CONTEMPORARY_OF]->(a2:Artist)
    RETURN a1.name, a2.name, r.period

• Rooms with descriptions:
  MATCH (s:Sala) WHERE s.description IS NOT NULL
    RETURN s.palau, s.id, s.description

• Themes in each palace:
  MATCH (p:Palau)-[:FEATURES_THEME]->(t:Theme)
    RETURN p.id as palau, t.name as theme



In [ ]:
# PART 2: Python Query Helper Class
# ===================================
from neo4j import GraphDatabase
import os
from tabulate import tabulate

class MuseumGraphDB:
    """Helper class to query the museum knowledge graph"""
    
    def __init__(self, uri="bolt://localhost:7687", user="neo4j", password=None):
        if password is None:
            password = os.getenv("password")
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
    
    def query(self, cypher):
        """Execute a custom Cypher query"""
        with self.driver.session() as session:
            return session.run(cypher).data()
    
    def get_artworks(self, limit=20):
        """Get all artworks with details"""
        result = self.query(f"""
        MATCH (a:ArtPiece)-[:LOCATED_IN]->(s:Sala)
        RETURN a.title as Title, a.artist as Artist, s.id as Room, a.dating as Dating, a.technique as Technique
        LIMIT {limit}
        """)
        return result
    
    def get_artworks_by_room(self, palau, sala):
        """Get artworks in a specific room"""
        result = self.query(f"""
        MATCH (a:ArtPiece)-[:LOCATED_IN]->(s:Sala {{id: '{sala}', palau: '{palau}'}})
        RETURN a.title as Title, a.artist as Artist, a.dating as Dating
        """)
        return result
    
    def get_room_description(self, palau, sala):
        """Get description of a room"""
        result = self.query(f"""
        MATCH (s:Sala {{id: '{sala}', palau: '{palau}'}})
        RETURN s.description as Description
        """)
        return result[0]['Description'] if result else None
    
    def get_artist_works(self, artist_name):
        """Get all works by an artist"""
        result = self.query(f"""
        MATCH (art:Artist {{name: '{artist_name}'}})-[:CREATED_BY]-(a:ArtPiece)
        RETURN a.title as Title, a.dating as Dating, a.technique as Technique
        """)
        return result
    
    def get_contemporaries(self, artist_name):
        """Get artists contemporary with a given artist"""
        result = self.query(f"""
        MATCH (a:Artist {{name: '{artist_name}'}})-[:CONTEMPORARY_OF]->(contemporary:Artist)
        RETURN contemporary.name as Artist
        """)
        return result
    
    def get_themes_in_palau(self, palau):
        """Get all themes in a palace"""
        result = self.query(f"""
        MATCH (p:Palau {{id: '{palau}'}})-[:FEATURES_THEME]->(t:Theme)
        RETURN t.name as Theme, t.description as Description
        """)
        return result
    
    def close(self):
        self.driver.close()

# Initialize the database connection
db = MuseumGraphDB()
print("✅ Database connection established!")


In [ ]:
# PART 3: Practical Query Examples
# ==================================

print("\n" + "="*70)
print("📚 MUSEUM KNOWLEDGE GRAPH - QUERY EXAMPLES")
print("="*70)

# Example 1: List all artworks
print("\n1️⃣  ALL ARTWORKS (first 10):")
print("-" * 70)
artworks = db.get_artworks(limit=10)
if artworks:
    for i, art in enumerate(artworks, 1):
        print(f"{i}. {art['Title']} by {art['Artist']} | Room {art['Room']} | {art['Dating']}")
else:
    print("No artworks found")

# Example 2: Get artworks in a specific room
print("\n2️⃣  ARTWORKS IN PALAU 1, SALA 0:")
print("-" * 70)
room_artworks = db.get_artworks_by_room('1', '0')
if room_artworks:
    for art in room_artworks:
        print(f"• {art['Title']} by {art['Artist']} ({art['Dating']})")
else:
    print("No artworks found in this room")

# Example 3: Get room description
print("\n3️⃣  ROOM DESCRIPTION (Palau 1, Sala 0):")
print("-" * 70)
description = db.get_room_description('1', '0')
if description:
    print(description[:300] + "..." if len(description) > 300 else description)
else:
    print("No description available")

# Example 4: Get themes
print("\n4️⃣  THEMES IN PALAU 1:")
print("-" * 70)
themes = db.get_themes_in_palau('1')
if themes:
    for theme in themes:
        print(f"• {theme['Theme']}")
        if theme['Description']:
            print(f"  → {theme['Description'][:150]}...")
else:
    print("No themes found")

# Example 5: Count statistics
print("\n5️⃣  DATABASE STATISTICS:")
print("-" * 70)
stats = db.query("""
MATCH (n) 
RETURN 
    labels(n)[0] as NodeType,
    COUNT(n) as Count
""")
for stat in stats:
    print(f"• {stat['NodeType']}: {stat['Count']} nodes")

print("\n" + "="*70)
print("✨ Try modifying the queries above to explore your data!")
print("="*70)


In [ ]:
# PART 4: Advanced Queries & Custom Search
# ==========================================

def search_artworks(keyword):
    """Search artworks by title or artist"""
    query = f"""
    MATCH (a:ArtPiece)
    WHERE a.title CONTAINS '{keyword}' OR a.artist CONTAINS '{keyword}'
    RETURN a.title as Title, a.artist as Artist, a.dating as Dating, a.technique as Technique
    LIMIT 10
    """
    return db.query(query)

def find_artist_collaborations(artist_name):
    """Find other artists who worked during the same period"""
    query = f"""
    MATCH (a1:Artist {{name: '{artist_name}'}})-[:CREATED_BY]-(art1:ArtPiece),
          (a2:Artist)-[:CREATED_BY]-(art2:ArtPiece)
    WHERE a1 <> a2 
          AND art1.dating = art2.dating 
          AND art1.dating IS NOT NULL
    RETURN DISTINCT a2.name as Collaborator, art1.dating as Period
    """
    return db.query(query)

def get_artistic_technique_stats():
    """Get statistics on artistic techniques"""
    query = """
    MATCH (a:ArtPiece)
    WHERE a.technique IS NOT NULL
    RETURN a.technique as Technique, COUNT(a) as Count
    ORDER BY Count DESC
    LIMIT 15
    """
    return db.query(query)

# Interactive search examples
print("\n" + "="*70)
print("🔍 TRY THESE SEARCHES")
print("="*70)

# Example: Search for a keyword
print("\n📌 Searching for artworks with 'retaule' in title:")
results = search_artworks('retaule')
print(f"Found {len(results)} results")
if results:
    for r in results[:3]:
        print(f"  • {r['Title']} by {r['Artist']}")

# Example: Get technique statistics
print("\n📌 Most common artistic techniques:")
techniques = get_artistic_technique_stats()
for i, t in enumerate(techniques[:5], 1):
    print(f"  {i}. {t['Technique']}: {t['Count']} artworks")

print("\n" + "="*70)
print("💡 Ready to use:")
print("  • db.query('CYPHER QUERY HERE') - Run any Cypher query")
print("  • search_artworks('keyword') - Search by title or artist")
print("  • find_artist_collaborations('Artist Name') - Find related artists")
print("  • get_artistic_technique_stats() - View technique statistics")
print("="*70)


In [ ]:
# PART 5: Save & Backup Your Knowledge Graph (Neo4j Native)
# ============================================================

print("\n" + "="*70)
print("NEO4J NATIVE BACKUP")
print("="*70)

print("""
SAVE YOUR KNOWLEDGE GRAPH:

1. From Terminal:
   $ neo4j stop
   $ neo4j-admin database dump neo4j GuIA/KG/neo4j.dump
   $ neo4j start

2. Verify the dump file:
   $ ls -lh GuIA/KG/neo4j.dump

3. The dump is now ready for:
   • Version control (git)
   • Backup to cloud storage
   • Sharing with team members
   • Deployment to production
""")

print("="*70)
print("RESTORE FROM BACKUP:")
print("="*70)

print("""
1. Stop Neo4j:
   $ neo4j stop

2. Load the backup:
   $ neo4j-admin database load neo4j GuIA/KG/neo4j.dump --overwrite-existing

3. Start Neo4j:
   $ neo4j start

4. Verify the data is restored:
   • Open http://localhost:7474
   • Run: MATCH (n) RETURN COUNT(n) as total_nodes
""")

print("="*70)
print("GIT WORKFLOW:")
print("="*70)

print("""
1. Create .gitignore for large files (optional):
   # GuIA/KG/.gitignore
   neo4j.dump  # If dump files are too large

2. Commit this notebook to git:
   $ git add GuIA/KG/kg.ipynb
   $ git commit -m "Add museum knowledge graph queries"

3. If committing backups:
   $ git add GuIA/KG/neo4j.dump
   $ git commit -m "Update knowledge graph backup"
""")

In [ ]:
# PART 6: Database Status Check
# ===============================

print("\n" + "="*70)
print("CURRENT DATABASE STATUS")
print("="*70)

# Check what's in the database
try:
    stats = db.query("""
    MATCH (n) 
    RETURN 
        labels(n)[0] as NodeType,
        COUNT(n) as Count
    ORDER BY Count DESC
    """)
    
    print("\nDatabase Contents:")
    total_nodes = 0
    for stat in stats:
        print(f"   • {stat['NodeType']}: {stat['Count']} nodes")
        total_nodes += stat['Count']
    
    print(f"\n   Total Nodes: {total_nodes}")
    
    # Check relationships
    rels = db.query("MATCH ()-[r]->() RETURN TYPE(r) as RelType, COUNT(r) as Count ORDER BY Count DESC")
    if rels:
        print("\nRelationships:")
        total_rels = 0
        for rel in rels:
            print(f"   • {rel['RelType']}: {rel['Count']} relationships")
            total_rels += rel['Count']
        print(f"\n   Total Relationships: {total_rels}")
    
except Exception as e:
    print(f"\nError checking database: {e}")

print("\n" + "="*70)
print("BACKUP LOCATION")
print("="*70)
print("""
When you run the backup command:
   neo4j-admin database dump neo4j GuIA/KG/neo4j.dump

The dump file will be created at:
   GuIA/KG/neo4j.dump

File details:
   • Size: Depends on data volume
   • Format: Binary native Neo4j format
   • Can be compressed: .dump can be bzipped for storage
   • Ready for: Git, S3, backup services, team sharing
""")
print("="*70)

In [ ]:
# PART 7: Removed experimental scraper/parser cell
# The previous scraping helper has been removed to keep the notebook clean.


In [15]:
# PART 8: Data Integration - Import Artists, Techniques, and Descriptions
# ===========================================================================
import pandas as pd
from neo4j import GraphDatabase
import os
from dotenv import load_dotenv

load_dotenv()
uri = "bolt://localhost:7687"
user = "neo4j"
pwd = os.getenv("password")

def update_artists_from_csv(file_path, uri, user, pwd):
    """Update existing Artist nodes with biographical information from AuthorInfo.csv"""
    try:
        # Read CSV with semicolon delimiter and latin-1 encoding
        df = pd.read_csv(file_path, delimiter=';', encoding='latin-1')
        
        driver = GraphDatabase.driver(uri, auth=(user, pwd))
        
        print(f"\n{'='*70}")
        print("UPDATING ARTIST NODES FROM CSV")
        print(f"{'='*70}")
        
        updated_count = 0
        skipped_count = 0
        
        with driver.session() as session:
            for _, row in df.iterrows():
                author_name = str(row["Author's Name"]).strip()
                information = str(row["Information"]).strip() if "Information" in row else ""
                
                if not author_name or not information or information == "None":
                    skipped_count += 1
                    continue
                
                # MERGE to find existing Artist by name, then SET biography
                result = session.run("""
                MERGE (a:Artist {name: $name})
                SET a.biography = $info
                RETURN a.name as name
                """, name=author_name, info=information)
                
                record = result.single()
                if record:
                    updated_count += 1
                    print(f"✓ Updated: {author_name[:40]}...")
        
        driver.close()
        
        print(f"\nSummary:")
        print(f"  • Artists updated: {updated_count}")
        print(f"  • Rows skipped: {skipped_count}")
        print(f"{'='*70}\n")
        
    except Exception as e:
        print(f"Error updating artists: {e}")

def create_techniques_from_csv(file_path, uri, user, pwd):
    """Create Technique nodes with information from Tech.csv"""
    try:
        # Read CSV with semicolon delimiter and latin-1 encoding
        df = pd.read_csv(file_path, delimiter=';', encoding='latin-1')
        
        driver = GraphDatabase.driver(uri, auth=(user, pwd))
        
        print(f"{'='*70}")
        print("CREATING TECHNIQUE NODES FROM CSV")
        print(f"{'='*70}")
        
        created_count = 0
        skipped_count = 0
        
        with driver.session() as session:
            for _, row in df.iterrows():
                technique_name = str(row["Art Technique"]).strip()
                information = str(row["Information"]).strip() if "Information" in row else ""
                
                if not technique_name:
                    skipped_count += 1
                    continue
                
                # MERGE Technique by name, SET information if provided
                if information and information.lower() != "blank" and information != "None":
                    result = session.run("""
                    MERGE (t:Technique {name: $name})
                    SET t.description = $info
                    RETURN t.name as name
                    """, name=technique_name, info=information)
                else:
                    result = session.run("""
                    MERGE (t:Technique {name: $name})
                    RETURN t.name as name
                    """, name=technique_name)
                
                record = result.single()
                if record:
                    created_count += 1
                    status = "✓ Created/Updated" if information and information.lower() != "blank" and information != "None" else "✓ Created"
                    print(f"{status}: {technique_name[:50]}...")
        
        driver.close()
        
        print(f"\nSummary:")
        print(f"  • Techniques processed: {created_count}")
        print(f"  • Rows skipped: {skipped_count}")
        print(f"{'='*70}\n")
        
    except Exception as e:
        print(f"Error creating techniques: {e}")

def add_artwork_descriptions(file_path, uri, user, pwd):
    """Add descriptions to ArtPiece nodes from Excel DESCRIPTION column"""
    try:
        # Read Excel file
        df = pd.read_excel(file_path, header=None)
        df = df[1:]  # skip title row
        df.columns = df.iloc[0]  # set headers
        df = df[1:]  # skip header row
        
        driver = GraphDatabase.driver(uri, auth=(user, pwd))
        
        print(f"{'='*70}")
        print("ADDING ARTWORK DESCRIPTIONS FROM EXCEL")
        print(f"{'='*70}")
        
        updated_count = 0
        skipped_count = 0
        
        with driver.session() as session:
            for _, row in df.iterrows():
                title = row['TÍTOL / DESCRIPCIÓ'] if 'TÍTOL / DESCRIPCIÓ' in row else None
                description = row['DESCRIPTION'] if 'DESCRIPTION' in row else None
                
                if not title or not description or pd.isna(description) or not str(description).strip():
                    skipped_count += 1
                    continue
                
                # MERGE ArtPiece by title, SET description
                result = session.run("""
                MERGE (a:ArtPiece {title: $title})
                SET a.description = $description
                RETURN a.title as title
                """, title=str(title).strip(), description=str(description).strip())
                
                record = result.single()
                if record:
                    updated_count += 1
                    print(f"✓ Added description: {title[:45]}...")
        
        driver.close()
        
        print(f"\nSummary:")
        print(f"  • Descriptions added: {updated_count}")
        print(f"  • Artworks without descriptions: {skipped_count}")
        print(f"{'='*70}\n")
        
    except Exception as e:
        print(f"Error adding descriptions: {e}")

# Run the import functions
print("\n")
update_artists_from_csv(
    "../raw_data/AuthorInfo.csv",
    uri, user, pwd
)

print("\n")
create_techniques_from_csv(
    "../raw_data/Tech.csv",
    uri, user, pwd
)

print("\n")
add_artwork_descriptions(
    "../raw_data/2026_obres_Museu_del_Renaixement.xlsx",
    uri, user, pwd
)




UPDATING ARTIST NODES FROM CSV
✓ Updated: Anton van Dasshort Mor...
✓ Updated: Perot Gascó...
✓ Updated: Mestre de Pau i Bernabeu...
✓ Updated: Mestre del Papagai...
✓ Updated: Hans Schwarts...
✓ Updated: Leone Leoni...
✓ Updated: Giacome Trezzo...
✓ Updated: Giovanni de Rossi...
✓ Updated: Jacob Matham...
✓ Updated: Lucas Kilian...
✓ Updated: Albrecht Dürer...
✓ Updated: Jan Wierix...
✓ Updated: Thomas Leu...
✓ Updated: Leonardo da Vinci...

Summary:
  • Artists updated: 14
  • Rows skipped: 0



CREATING TECHNIQUE NODES FROM CSV
✓ Created: Pintura, oli sobre fusta...
✓ Created: Pintura, tremp i oli sobre fusta...
✓ Created: Pintura sobre fusta...
✓ Created: Escultura, fusta policromada...
✓ Created: Escultura, talla en fusta daurada i policromada...
✓ Created: Escultura, fusta daurada i policromada...
✓ Created: Escultura, talla en gres tipus Montjuïc...
✓ Created: Escultura, talla en pedra sorrenca de Montjuïc...
✓ Created: Escultura, relleu tallat en marbre blanc...
✓ Created: E

In [16]:
# VERIFICATION: Check what was imported
from neo4j import GraphDatabase
import os
from dotenv import load_dotenv

load_dotenv()
uri = "bolt://localhost:7687"
user = "neo4j"
pwd = os.getenv("password")

driver = GraphDatabase.driver(uri, auth=(user, pwd))

print("\n" + "="*70)
print("VERIFICATION - Data Import Results")
print("="*70)

with driver.session() as session:
    # Check artists with biography
    result = session.run("""
    MATCH (a:Artist) WHERE a.biography IS NOT NULL
    RETURN COUNT(a) as count, COLLECT(a.name)[0..5] as sample
    """)
    row = result.single()
    if row:
        print(f"\nArtists with biography: {row['count']}")
        print(f"  Sample: {row['sample']}")
    
    # Check technique nodes created
    result = session.run("""
    MATCH (t:Technique)
    RETURN COUNT(t) as count, COLLECT(t.name)[0..5] as sample
    """)
    row = result.single()
    if row:
        print(f"\nTechnique nodes created: {row['count']}")
        print(f"  Sample: {row['sample']}")
    
    # Check artpieces with descriptions
    result = session.run("""
    MATCH (a:ArtPiece) WHERE a.description IS NOT NULL
    RETURN COUNT(a) as count, COLLECT(a.title)[0..5] as sample
    """)
    row = result.single()
    if row:
        print(f"\nArtworks with descriptions: {row['count']}")
        print(f"  Sample: {row['sample']}")
    
    # Total database status
    result = session.run("""
    MATCH (n) 
    RETURN 
        labels(n)[0] as NodeType,
        COUNT(n) as Count
    ORDER BY Count DESC
    """)
    print(f"\nCurrent database status:")
    for row in result:
        print(f"  • {row['NodeType']}: {row['Count']}")

driver.close()
print("\n" + "="*70)


VERIFICATION - Data Import Results

Artists with biography: 14
  Sample: ['Anton van Dasshort Mor', 'Perot Gascó', 'Mestre de Pau i Bernabeu', 'Mestre del Papagai', 'Hans Schwarts']

Technique nodes created: 40
  Sample: ['Pintura, oli sobre fusta', 'Pintura, tremp i oli sobre fusta', 'Pintura sobre fusta', 'Escultura, fusta policromada', 'Escultura, talla en fusta daurada i policromada']

Artworks with descriptions: 60
  Sample: ["Retrat de cavaller de l'orde de sant Jaume", 'Mare de Déu amb el Nen ("Madonna di Trapani")', "Bust reliquiari d'una santa", 'Sant Jeroni penitent', 'Davallament']

Current database status:
  • ArtPiece: 72
  • Technique: 40
  • Theme: 38
  • Artist: 23
  • Sala: 6
  • Palau: 3



In [14]:
# DEBUG: Check encoding and CSV data
import pandas as pd
from neo4j import GraphDatabase
import os
from dotenv import load_dotenv

load_dotenv()
uri = "bolt://localhost:7687"
user = "neo4j"
pwd = os.getenv("password")

print("\n" + "="*70)
print("DEBUG: Checking CSV encoding and data")
print("="*70)

# Try different encodings
encodings_to_try = ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252', 'utf-16']

for encoding in encodings_to_try:
    try:
        print(f"\nTrying encoding: {encoding}")
        df = pd.read_csv("../raw_data/AuthorInfo.csv", delimiter=';', encoding=encoding)
        print(f"  ✓ Success! Read {len(df)} rows")
        print(f"  Columns: {df.columns.tolist()}")
        print(f"  First author: '{df.iloc[0, 0] if len(df) > 0 else 'N/A'}'")
        auth_encoding = encoding
        break
    except Exception as e:
        print(f"  ✗ Failed: {type(e).__name__}")

print("\n" + "-"*70)

for encoding in encodings_to_try:
    try:
        print(f"Trying encoding: {encoding}")
        df = pd.read_csv("../raw_data/Tech.csv", delimiter=';', encoding=encoding)
        print(f"  ✓ Success! Read {len(df)} rows")
        print(f"  Columns: {df.columns.tolist()}")
        print(f"  First technique: '{df.iloc[0, 0] if len(df) > 0 else 'N/A'}'")
        tech_encoding = encoding
        break
    except Exception as e:
        print(f"  ✗ Failed: {type(e).__name__}")


DEBUG: Checking CSV encoding and data

Trying encoding: utf-8
  ✗ Failed: UnicodeDecodeError

Trying encoding: latin-1
  ✓ Success! Read 14 rows
  Columns: ["Author's Name", 'Information']
  First author: 'Anton van Dasshort Mor'

----------------------------------------------------------------------
Trying encoding: utf-8
  ✗ Failed: UnicodeDecodeError
Trying encoding: latin-1
  ✓ Success! Read 40 rows
  Columns: ['Art Technique', 'Information']
  First technique: 'Pintura, oli sobre fusta'


In [18]:
# PART 9: Display Examples of Imported Data
# ==========================================
from neo4j import GraphDatabase
import os
from dotenv import load_dotenv

load_dotenv()
uri = "bolt://localhost:7687"
user = "neo4j"
pwd = os.getenv("password")

driver = GraphDatabase.driver(uri, auth=(user, pwd))

print("\n" + "="*70)
print("IMPORTED DATA EXAMPLES")
print("="*70)

with driver.session() as session:
    # Show artists with biographies
    print("\n1. ARTISTS WITH BIOGRAPHY:")
    print("-" * 70)
    result = session.run("""
    MATCH (a:Artist) WHERE a.biography IS NOT NULL
    RETURN a.name as name, size(a.biography) as bio_length
    ORDER BY a.name
    LIMIT 5
    """)
    for row in result:
        print(f"• {row['name']}")
        print(f"  Biography length: {row['bio_length']} characters\n")
    
    # Show techniques
    print("\n2. TECHNIQUES IN KNOWLEDGE GRAPH:")
    print("-" * 70)
    result = session.run("""
    MATCH (t:Technique)
    RETURN t.name as name
    ORDER BY t.name
    LIMIT 10
    """)
    for i, row in enumerate(result, 1):
        print(f"{i:2}. {row['name']}")
    
    # Show artworks with descriptions
    print("\n3. ARTWORKS WITH DESCRIPTIONS (Sample):")
    print("-" * 70)
    result = session.run("""
    MATCH (a:ArtPiece) WHERE a.description IS NOT NULL
    RETURN a.title as title, a.description as desc
    ORDER BY a.title
    LIMIT 3
    """)
    for i, row in enumerate(result, 1):
        desc = row['desc'][:100] + "..." if len(row['desc']) > 100 else row['desc']
        print(f"{i}. {row['title']}")
        print(f"   {desc}\n")

driver.close()

print("="*70)
print("NEXT STEPS:")
print("="*70)
print("""
You can now:

1. Link techniques to artworks:
   MATCH (a:ArtPiece) WHERE a.technique IS NOT NULL
   MATCH (t:Technique) WHERE t.name = a.technique
   MERGE (a)-[:USES_TECHNIQUE]->(t)

2. Query artworks by technique:
   MATCH (a:ArtPiece)-[:USES_TECHNIQUE]->(t:Technique)
   RETURN a.title, t.name, a.description

3. Query artist information:
   MATCH (a:Artist) WHERE a.biography IS NOT NULL
   RETURN a.name, a.biography
""")
print("="*70)


IMPORTED DATA EXAMPLES

1. ARTISTS WITH BIOGRAPHY:
----------------------------------------------------------------------
• Albrecht Dürer
  Biography length: 1291 characters

• Anton van Dasshort Mor
  Biography length: 7817 characters

• Giacome Trezzo
  Biography length: 5546 characters

• Giovanni de Rossi
  Biography length: 4619 characters

• Hans Schwarts
  Biography length: 3038 characters


2. TECHNIQUES IN KNOWLEDGE GRAPH:
----------------------------------------------------------------------
 1. Acer daurat
 2. Ceràmica
 3. Ceràmica vidriada
 4. Escultura, fusta daurada i policromada
 5. Escultura, fusta policromada
 6. Escultura, marbre blanc. Placa rectangular
 7. Escultura, relleu tallat en marbre blanc
 8. Escultura, talla en fusta daurada i policromada
 9. Escultura, talla en gres
10. Escultura, talla en gres tipus Montjuïc

3. ARTWORKS WITH DESCRIPTIONS (Sample):
----------------------------------------------------------------------
1. Anunciació
   One of the episode

In [19]:
# PART 10: Link Techniques to Artworks
# =====================================
from neo4j import GraphDatabase
import os
from dotenv import load_dotenv

load_dotenv()
uri = "bolt://localhost:7687"
user = "neo4j"
pwd = os.getenv("password")

def link_techniques_to_artworks(uri, user, pwd):
    """Create USES_TECHNIQUE relationships between ArtPiece and Technique nodes"""
    try:
        driver = GraphDatabase.driver(uri, auth=(user, pwd))
        
        print(f"\n{'='*70}")
        print("LINKING TECHNIQUES TO ARTWORKS")
        print(f"{'='*70}")
        
        with driver.session() as session:
            # First, check how many artworks have technique information
            result = session.run("""
            MATCH (a:ArtPiece) WHERE a.technique IS NOT NULL
            RETURN COUNT(a) as count
            """)
            artworks_with_tech = result.single()['count']
            print(f"\nArtworks with technique information: {artworks_with_tech}")
            
            # Create relationships
            result = session.run("""
            MATCH (a:ArtPiece) WHERE a.technique IS NOT NULL
            MATCH (t:Technique) WHERE t.name = a.technique
            MERGE (a)-[:USES_TECHNIQUE]->(t)
            RETURN COUNT(*) as relationships_created
            """)
            
            created = result.single()['relationships_created']
            print(f"USES_TECHNIQUE relationships created: {created}")
            
            # Show some examples
            print(f"\nExamples of linked artwork-technique pairs:")
            print("-" * 70)
            result = session.run("""
            MATCH (a:ArtPiece)-[:USES_TECHNIQUE]->(t:Technique)
            RETURN a.title as artwork, t.name as technique
            LIMIT 10
            """)
            
            for i, row in enumerate(result, 1):
                print(f"{i:2}. {row['artwork'][:40]:<40} → {row['technique'][:30]}")
            
            # Check for unmatched techniques
            result = session.run("""
            MATCH (a:ArtPiece) WHERE a.technique IS NOT NULL
            WITH a.technique as tech_name
            WHERE NOT EXISTS { MATCH (t:Technique) WHERE t.name = tech_name }
            RETURN DISTINCT tech_name
            ORDER BY tech_name
            """)
            
            unmatched = list(result)
            if unmatched:
                print(f"\nTechniques not matched to Technique nodes ({len(unmatched)}):")
                print("-" * 70)
                for row in unmatched[:5]:
                    print(f"  • {row['tech_name']}")
                if len(unmatched) > 5:
                    print(f"  ... and {len(unmatched) - 5} more")
            else:
                print(f"\nAll artwork techniques successfully matched!")
        
        driver.close()
        print(f"\n{'='*70}\n")
        
    except Exception as e:
        print(f"Error linking techniques: {e}")

# Run the linking function
link_techniques_to_artworks(uri, user, pwd)


LINKING TECHNIQUES TO ARTWORKS

Artworks with technique information: 72
USES_TECHNIQUE relationships created: 68

Examples of linked artwork-technique pairs:
----------------------------------------------------------------------
 1. Retrat de cavaller de l'orde de sant Jau → Pintura, oli sobre fusta
 2. Mare de Déu de la Llet                   → Pintura, oli sobre fusta
 3. Crist amb la Creu                        → Pintura, oli sobre fusta
 4. Epifania                                 → Pintura, oli sobre fusta
 5. Anunciació                               → Pintura, oli sobre fusta
 6. La Visitació                             → Pintura, oli sobre fusta
 7. Davallament                              → Pintura, oli sobre fusta
 8. Sant Jeroni penitent                     → Pintura, oli sobre fusta
 9. Flagel·lació de Crist                    → Pintura, oli sobre fusta
10. Retrat de Carles V                       → Pintura, oli sobre fusta

Techniques not matched to Technique nodes (1):
--

In [20]:
# PART 11: Add Missing Technique - Medalla
# ==========================================
from neo4j import GraphDatabase
import os
from dotenv import load_dotenv

load_dotenv()
uri = "bolt://localhost:7687"
user = "neo4j"
pwd = os.getenv("password")

driver = GraphDatabase.driver(uri, auth=(user, pwd))

print(f"\n{'='*70}")
print("ADDING MISSING TECHNIQUE: Medalla")
print(f"{'='*70}")

with driver.session() as session:
    # Create the missing Medalla technique
    result = session.run("""
    MERGE (t:Technique {name: 'Medalla'})
    RETURN t.name as name
    """)
    
    print("\nCreated Technique node:")
    print(f"  • Medalla")
    
    # Now link it to artworks that use this technique
    result = session.run("""
    MATCH (a:ArtPiece) WHERE a.technique = 'Medalla'
    MATCH (t:Technique {name: 'Medalla'})
    MERGE (a)-[:USES_TECHNIQUE]->(t)
    RETURN COUNT(*) as linked
    """)
    
    linked = result.single()['linked']
    print(f"\nLinked artworks: {linked}")
    
    # Show the linked artwork
    result = session.run("""
    MATCH (a:ArtPiece)-[:USES_TECHNIQUE]->(t:Technique {name: 'Medalla'})
    RETURN a.title as title, a.artist as artist, a.dating as dating
    """)
    
    print("\nArtwork using Medalla technique:")
    for row in result:
        print(f"  • {row['title']}")
        print(f"    Artist: {row['artist']}")
        print(f"    Dating: {row['dating']}")
    
    # Final verification
    result = session.run("""
    MATCH (a:ArtPiece) WHERE a.technique IS NOT NULL
    WITH a.technique as tech_name
    WHERE NOT EXISTS { MATCH (t:Technique) WHERE t.name = tech_name }
    RETURN COUNT(DISTINCT tech_name) as unmatched_count
    """)
    
    unmatched = result.single()['unmatched_count']
    print(f"\nRemaining unmatched techniques: {unmatched}")
    
    if unmatched == 0:
        print("\n✓ SUCCESS! All artwork techniques are now linked to Technique nodes!")
    
    # Show final statistics
    result = session.run("""
    MATCH (a:ArtPiece)-[:USES_TECHNIQUE]->(t:Technique)
    RETURN COUNT(DISTINCT a) as total_artworks, COUNT(DISTINCT t) as total_techniques
    """)
    
    stats = result.single()
    print(f"\nFinal Statistics:")
    print(f"  • Artworks with techniques: {stats['total_artworks']}")
    print(f"  • Unique techniques: {stats['total_techniques']}")

driver.close()
print(f"\n{'='*70}\n")


ADDING MISSING TECHNIQUE: Medalla

Created Technique node:
  • Medalla

Linked artworks: 4

Artwork using Medalla technique:
  • Pius V. Commemoració de la victòria naval de Lepanto
    Artist: Giovanni de Rossi
    Dating: 1571
  • Carles V adult i la batalla de Mülberg
    Artist: Leone Leoni
    Dating: 1549
  • Felip II. Commemoració de l'abdicació de Carles V
    Artist: Giacome Trezzo
    Dating: 1555
  • Carles V jove i la divista Plus Oultre
    Artist: Hans Schwarts
    Dating: c. 1521

Remaining unmatched techniques: 0

✓ SUCCESS! All artwork techniques are now linked to Technique nodes!

Final Statistics:
  • Artworks with techniques: 72
  • Unique techniques: 33


